# Analisi Classificazione tramite Regex

Questo notebook implementa una classificazione binaria *AI* / *NON AI* utilizzando un'espressione regolare mirata. L'elaborazione avviene su **10 thread simultanei** e processa i file CSV in modalità **lazy (a chunk)** per garantire un occupazione di memoria quasi nulla (Zero Memory Footprint).

Alla fine viene generata una **matrice di confusione** aggregando i risultati distribuiti per confrontare la predizione della Regex contro la colonna `CLASSIFICAZIONE` pre-calcolata.

In [11]:
import pandas as pd
import numpy as np
import re
import glob
import os
import csv
import hashlib
from concurrent.futures import ThreadPoolExecutor, as_completed
import matplotlib.pyplot as plt
import seaborn as sns

## 1. Setup & Regex

In [12]:
# Configurazione percorsi e thread
DATA_DIR = '../data'
FILE_PATTERN = 'classified_multiclass_aiuti_*.csv'
MAX_WORKERS = 10
CHUNK_SIZE = 100000

# Regex per rilevare AI (minimizzando i falsi positivi)
# Usiamo (?i:...) per rendere l'elenco principale case-insensitive, pur mantenendo AI e IA strettamente case-sensitive per non prendere le preposizioni.
AI_REGEX_PATTERN = r'(?i:\b(intelligenza artificiale|robotica collaborativa|visione artificiale|algoritmi predittivi|manutenzione preventiva|predictive analytics|artificial intelligence|machine learning|deep learning|reti neurali|computer vision|ai generativa|elaborazione del linguaggio naturale|nlp|llm|virtual assistant|virtual assistants|realtà virtuale|realtà aumentata|virtual reality|augmented reality|guida autonoma|chatbot|classificazione semantica)\b)|\b(IA|tecnologie AI|tecnologie IA|modelli di AI|algoritmi di AI|algoritmi di IA|LLM)\b'
ai_regex = re.compile(AI_REGEX_PATTERN)

## 2. Funzione di Processing (Lazy)

In [13]:
def process_file(filepath):
    print(f"Elaborazione avviata: {os.path.basename(filepath)}")
    
    # Inizializza contatori (True Positive, False Positive, True Negative, False Negative)
    # Assumiamo AI = Positive Class
    tp, fp, tn, fn = 0, 0, 0, 0
    
    tp_conf_all, fp_conf_all, tn_conf_all, fn_conf_all = [], [], [], []
    tp_dict, fp_dict, tn_dict, fn_dict = {}, {}, {}, {}
    
    tp_records = pd.DataFrame()
    top_fn_records = pd.DataFrame()
    top_fp_records = pd.DataFrame()
    low_conf_ai_records = pd.DataFrame()
    
    try:
        # Lettura Lazy Chunked. Vengono estratte unicamente le colonne necessarie al processing.
        chunk_iter = pd.read_csv(
            filepath, 
            chunksize=CHUNK_SIZE, 
            usecols=['DESCRIZIONE_PROGETTO', 'CLASSIFICAZIONE', 'CLASSIFICAZIONE_CONFIDENZA'],
            dtype=str
        )
        
        for chunk in chunk_iter:
            # Sanitizzazione e parsing stringhe base
            desc = chunk['DESCRIZIONE_PROGETTO'].fillna('')
            current_classification = chunk['CLASSIFICAZIONE'].fillna('NON_AI').str.upper()
            chunk['CLASSIFICAZIONE_CONFIDENZA'] = pd.to_numeric(chunk['CLASSIFICAZIONE_CONFIDENZA'], errors='coerce').fillna(0.0)
            
            # Predizione Regex: True se troviamo il pattern, False altrimenti
            regex_preds = desc.str.contains(ai_regex, regex=True).fillna(False)
            
            # Ground Truth (dal file CSV)
            true_labels = (current_classification == 'AI')
            
            # Update metriche (con maschere)
            tp_mask = (regex_preds == True) & (true_labels == True)
            fp_mask = (regex_preds == True) & (true_labels == False)
            tn_mask = (regex_preds == False) & (true_labels == False)
            fn_mask = (regex_preds == False) & (true_labels == True)
            
            tp += tp_mask.sum()
            fp += fp_mask.sum()
            tn += tn_mask.sum()
            fn += fn_mask.sum()
            
            # Somma delle confidenze e salvataggio per quantili
            conf = chunk['CLASSIFICAZIONE_CONFIDENZA']
            tp_conf_all.append(conf[tp_mask].values)
            fp_conf_all.append(conf[fp_mask].values)
            tn_conf_all.append(conf[tn_mask].values)
            fn_conf_all.append(conf[fn_mask].values)
            
            # Generazione Hash per tracciare elementi unici (footprint molto ridotto)
            hashes = desc.apply(lambda x: hashlib.md5(str(x).encode('utf-8')).hexdigest())
            
            # Mappa per descrizioni uniche (hash -> confidenza)
            tp_dict.update(dict(zip(hashes[tp_mask], conf[tp_mask])))
            fp_dict.update(dict(zip(hashes[fp_mask], conf[fp_mask])))
            tn_dict.update(dict(zip(hashes[tn_mask], conf[tn_mask])))
            fn_dict.update(dict(zip(hashes[fn_mask], conf[fn_mask])))
            
            # Estrazione False Negatives ad alta confidenza
            fn_hq_mask = fn_mask & (chunk['CLASSIFICAZIONE_CONFIDENZA'] > 0.9)
            if fn_hq_mask.any():
                top_fn_records = pd.concat([top_fn_records, chunk[fn_hq_mask]])
                
            # Estrazione True Positives per export dedicato
            if tp_mask.any():
                tp_records = pd.concat([tp_records, chunk[tp_mask]])
                
            # Estrazione False Positives ad alta confidenza
            fp_hq_mask = fp_mask & (chunk['CLASSIFICAZIONE_CONFIDENZA'] > 0.9)
            if fp_hq_mask.any():
                top_fp_records = pd.concat([top_fp_records, chunk[fp_hq_mask]])
                
            # Estrazione AI a bassa confidenza (mantenendo solo le top 20 uniche per operazione lazy)
            ai_mask = (true_labels == True)
            if ai_mask.any():
                ai_chunk = chunk[ai_mask]
                low_conf = ai_chunk.drop_duplicates(subset=['DESCRIZIONE_PROGETTO']).nsmallest(20, 'CLASSIFICAZIONE_CONFIDENZA')
                low_conf_ai_records = pd.concat([low_conf_ai_records, low_conf]).drop_duplicates(subset=['DESCRIZIONE_PROGETTO']).nsmallest(20, 'CLASSIFICAZIONE_CONFIDENZA')
            
    except Exception as e:
        print(f"Errore in {filepath}: {e}")
        return None
        
    print(f"Completato: {os.path.basename(filepath)} | TP:{tp} FP:{fp} TN:{tn} FN:{fn}")
    return {
        'file': os.path.basename(filepath),
        'tp': tp,        'fp': fp,
        'tn': tn,        'fn': fn,
        'tp_conf_all': tp_conf_all, 'fp_conf_all': fp_conf_all,
        'tn_conf_all': tn_conf_all, 'fn_conf_all': fn_conf_all,
        'tp_dict': tp_dict, 'fp_dict': fp_dict,
        'tn_dict': tn_dict, 'fn_dict': fn_dict,
        'tp_records': tp_records,
        'top_fns': top_fn_records,
        'top_fps': top_fp_records,
        'low_ai': low_conf_ai_records
    }

## 3. Multithreading Execution

In [14]:
files = glob.glob(os.path.join(DATA_DIR, FILE_PATTERN))
print(f"\nTrovati {len(files)} file per il processing concorrente su 10 thread.")

results = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    # Assegna i task ai worker nel thread pool
    futures = {executor.submit(process_file, f): f for f in files}
    
    for future in as_completed(futures):
        res = future.result()
        if res:
            results.append(res)

print("\nTutti i thread sono stati completati con successo.")


Trovati 0 file per il processing concorrente su 10 thread.

Tutti i thread sono stati completati con successo.


## 4. Aggregazione e Matrice di Confusione

In [15]:
if not results:
    print("Nessun risultato disponibile da processare.")
else:
    # Aggregazione Totali
    total_tp = sum(r['tp'] for r in results)
    total_fp = sum(r['fp'] for r in results)
    total_tn = sum(r['tn'] for r in results)
    total_fn = sum(r['fn'] for r in results)
    
    # Raggruppamento delle confidenze (totali)
    all_tp_conf = np.concatenate([arr for r in results for arr in r['tp_conf_all']]) if total_tp > 0 else np.array([])
    all_fp_conf = np.concatenate([arr for r in results for arr in r['fp_conf_all']]) if total_fp > 0 else np.array([])
    all_tn_conf = np.concatenate([arr for r in results for arr in r['tn_conf_all']]) if total_tn > 0 else np.array([])
    all_fn_conf = np.concatenate([arr for r in results for arr in r['fn_conf_all']]) if total_fn > 0 else np.array([])

    print("--- Risultati Totali Aggregati ---")
    print(f"Totale Record: {total_tp + total_fp + total_tn + total_fn:,}")
    print(f"True Positives (TP): {total_tp:,}")
    print(f"False Positives (FP): {total_fp:,}")
    print(f"True Negatives (TN): {total_tn:,}")
    print(f"False Negatives (FN): {total_fn:,}\n")
    
    def get_stats(arr):
        if len(arr) == 0: return 0, 0, 0
        return np.mean(arr), np.percentile(arr, 25), np.percentile(arr, 75)

    # Calcolo Statistiche Confidenza
    tp_mean, tp_q25, tp_q75 = get_stats(all_tp_conf)
    fp_mean, fp_q25, fp_q75 = get_stats(all_fp_conf)
    tn_mean, tn_q25, tn_q75 = get_stats(all_tn_conf)
    fn_mean, fn_q25, fn_q75 = get_stats(all_fn_conf)
    
    # Matrice di Confusione
    conf_matrix = np.array([
        [total_tn, total_fp], 
        [total_fn, total_tp]
    ])
    
    # Annotazioni Personalizzate (con media, Q1 e Q3)
    labels = np.array([
        [f"{total_tn:,}\nConf AVG: {tn_mean:.2f}\nQ1-Q3: [{tn_q25:.2f} - {tn_q75:.2f}]", 
         f"{total_fp:,}\nConf AVG: {fp_mean:.2f}\nQ1-Q3: [{fp_q25:.2f} - {fp_q75:.2f}]"],
        [f"{total_fn:,}\nConf AVG: {fn_mean:.2f}\nQ1-Q3: [{fn_q25:.2f} - {fn_q75:.2f}]", 
         f"{total_tp:,}\nConf AVG: {tp_mean:.2f}\nQ1-Q3: [{tp_q25:.2f} - {tp_q75:.2f}]"]
    ])
    
    # Plot
    plt.figure(figsize=(8, 6))
    sns.heatmap(
        conf_matrix, 
        annot=labels, 
        fmt='', 
        cmap='Blues', 
        xticklabels=['NON_AI (Pred: Regex)', 'AI (Pred: Regex)'], 
        yticklabels=['NON_AI (Reale: Model)', 'AI (Reale: Model)']
    )
    
    plt.title('Matrice di Confusione Totale: Regex VS Modello (CLASSIFICAZIONE)', pad=15)
    plt.ylabel('Ground Truth (dal dataset originale)')
    plt.xlabel('Valutazione Lazy Regex')
    plt.show()

Nessun risultato disponibile da processare.


## 4.1. Aggregazione e Matrice di Confusione (Solo Descrizioni Uniche)

Questa sezione analizza le performance della regex depurandole dalle voci doppie (progetti con la stessa identical `DESCRIZIONE_PROGETTO` ripetuta più volte nei vari CSV). VBS/CUP/anni). L'uso di un hash MD5 durante la lettura assicura un mapping esatto in Poca Memoria.

In [16]:
if not results:
    print("Nessun risultato disponibile da processare.")
else:
    # Unione dei dizionari tramite update per mantenere l'univocità degli hash
    all_tp_dict, all_fp_dict, all_tn_dict, all_fn_dict = {}, {}, {}, {}
    for r in results:
        all_tp_dict.update(r['tp_dict'])
        all_fp_dict.update(r['fp_dict'])
        all_tn_dict.update(r['tn_dict'])
        all_fn_dict.update(r['fn_dict'])
        
    uniq_tp = len(all_tp_dict)
    uniq_fp = len(all_fp_dict)
    uniq_tn = len(all_tn_dict)
    uniq_fn = len(all_fn_dict)
    
    print("--- Risultati Aggregati (Descrizioni Uniche) ---")
    print(f"Totale Record Unici: {uniq_tp + uniq_fp + uniq_tn + uniq_fn:,}")
    print(f"True Positives (TP): {uniq_tp:,}")
    print(f"False Positives (FP): {uniq_fp:,}")
    print(f"True Negatives (TN): {uniq_tn:,}")
    print(f"False Negatives (FN): {uniq_fn:,}\n")
    
    def get_stats_dict(d):
        if len(d) == 0: return 0, 0, 0
        arr = list(d.values())
        return np.mean(arr), np.percentile(arr, 25), np.percentile(arr, 75)
        
    # Calcolo Statistiche Confidenza per le uniche
    utp_mean, utp_q25, utp_q75 = get_stats_dict(all_tp_dict)
    ufp_mean, ufp_q25, ufp_q75 = get_stats_dict(all_fp_dict)
    utn_mean, utn_q25, utn_q75 = get_stats_dict(all_tn_dict)
    ufn_mean, ufn_q25, ufn_q75 = get_stats_dict(all_fn_dict)
    
    # Matrice di Confusione
    conf_matrix_uniq = np.array([
        [uniq_tn, uniq_fp], 
        [uniq_fn, uniq_tp]
    ])
    
    labels_uniq = np.array([
        [f"{uniq_tn:,}\nConf AVG: {utn_mean:.2f}\nQ1-Q3: [{utn_q25:.2f} - {utn_q75:.2f}]", 
         f"{uniq_fp:,}\nConf AVG: {ufp_mean:.2f}\nQ1-Q3: [{ufp_q25:.2f} - {ufp_q75:.2f}]"],
        [f"{uniq_fn:,}\nConf AVG: {ufn_mean:.2f}\nQ1-Q3: [{ufn_q25:.2f} - {ufn_q75:.2f}]", 
         f"{uniq_tp:,}\nConf AVG: {utp_mean:.2f}\nQ1-Q3: [{utp_q25:.2f} - {utp_q75:.2f}]"]
    ])
    
    # Plot
    plt.figure(figsize=(8, 6))
    sns.heatmap(
        conf_matrix_uniq, 
        annot=labels_uniq, 
        fmt='', 
        cmap='Greens', 
        xticklabels=['NON_AI (Pred: Regex)', 'AI (Pred: Regex)'], 
        yticklabels=['NON_AI (Reale: Model)', 'AI (Reale: Model)']
    )
    
    plt.title('Matrice di Confusione (Solo Descrizioni Uniche)', pad=15)
    plt.ylabel('Ground Truth (dal dataset originale)')
    plt.xlabel('Valutazione Lazy Regex')
    plt.show()

Nessun risultato disponibile da processare.


## 5. Analisi False Negatives ad Alta Confidenza

Analizziamo i progetti che il modello ha classificato come "AI" con un'elevata **CLASSIFICAZIONE_CONFIDENZA**, ma che la regex non è riuscita a catturare (Classificati preventivamente dalla regex come "NON_AI").

In [17]:
if results:
    # Aggrega tutti i dataframe dei false negatives trovati in ogni file
    all_fns = pd.concat([r['top_fns'] for r in results if 'top_fns' in r and not r['top_fns'].empty])
    
    if not all_fns.empty:
        # Ordina per confidenza discendente e visualizza i top 20
        top_fns_overall = all_fns.sort_values(by='CLASSIFICAZIONE_CONFIDENZA', ascending=True)
        
        print("I 20 progetti classificati 'AI' dal modello (con highest CONFIDENZA) ma mancati dalla regex:\n")
        for idx, row in top_fns_overall.head(20).iterrows():
            print(f"Confidenza: {row['CLASSIFICAZIONE_CONFIDENZA']:.4f}")
            desc = row['DESCRIZIONE_PROGETTO']
            if len(desc) > 1024: desc = desc[:1024] + "..."
            print(f"Descrizione: {desc}")
            print("-" * 80)
            
        # Esporta in CSV come richiesto (old = label modello, descrizione = desc. progetto)
        fn_export = all_fns[['CLASSIFICAZIONE', 'DESCRIZIONE_PROGETTO']].copy()
        fn_export.rename(columns={'CLASSIFICAZIONE': 'old', 'DESCRIZIONE_PROGETTO': 'descrizione'}, inplace=True)
        fn_export_path = os.path.join(DATA_DIR, 'regex_fn_mismatches.csv')
        fn_export.to_csv(fn_export_path, index=False, quoting=1)
        print(f"\n[+] Salvati {len(fn_export)} False Negatives (modello=AI, regex=NON_AI) in: {fn_export_path}")
        
    else:
        print("Nessun false negative trovato.")
else:
    print("Elaborazione non completata, risultati assenti.")

Elaborazione non completata, risultati assenti.


## 6. Analisi False Positives ad Alta Confidenza

Analizziamo i progetti che il modello ha classificato come "NON_AI" con un'elevata **CLASSIFICAZIONE_CONFIDENZA** (> 0.9), ma che la regex ha classificato come "AI" (Falsi Positivi).

In [18]:
if results:
    # Aggrega tutti i dataframe dei false positives trovati in ogni file
    all_fps = pd.concat([r['top_fps'] for r in results if 'top_fps' in r and not r['top_fps'].empty])
    
    if not all_fps.empty:
        # Ordina per confidenza discendente (già filtrati con confidenza > 0.9)
        top_fps_overall = all_fps.sort_values(by='CLASSIFICAZIONE_CONFIDENZA', ascending=False)
        print("I progetti classificati 'NON_AI' dal modello (con highest CONFIDENZA > 0.9) che la regex ha classificato 'AI':\n")
        
        for idx, row in top_fps_overall.head(20).iterrows():
            print(f"Confidenza: {row['CLASSIFICAZIONE_CONFIDENZA']:.4f}")
            desc = row['DESCRIZIONE_PROGETTO']
            if pd.notna(desc) and len(desc) > 500: desc = desc[:497] + "..."
            print(f"Descrizione: {desc}")
            print("-" * 80)
            
        # Esporta in CSV come richiesto (old = label modello, descrizione = desc. progetto)
        fp_export = all_fps[['CLASSIFICAZIONE', 'DESCRIZIONE_PROGETTO']].copy()
        fp_export.rename(columns={'CLASSIFICAZIONE': 'old', 'DESCRIZIONE_PROGETTO': 'descrizione'}, inplace=True)
        fp_export_path = os.path.join(DATA_DIR, 'regex_fp_mismatches_hq.csv')
        fp_export.to_csv(fp_export_path, index=False, quoting=1)
        print(f"\n[+] Salvati {len(fp_export)} False Positives ad alta confidenza (modello=NON_AI, regex=AI) in: {fp_export_path}")
        
    else:
        print("Nessun false positive con confidenza > 0.9 trovato.")
else:
    print("Elaborazione non completata, risultati assenti.")

Elaborazione non completata, risultati assenti.


## 7. Analisi Progetti AI a Bassa Confidenza

Analizziamo i progetti che il modello ha classificato come "AI" ma con il livello di **CLASSIFICAZIONE_CONFIDENZA** più basso in assoluto. Questo aiuta a comprendere su quali testi il modello si è dimostrato più incerto.

## 8. Esportazione True Positives

Esportiamo in CSV le descrizioni classificate come True Positive dalla regex, insieme alla loro label di classificazione impostata a `ai`.

In [19]:
if results:
    tp_frames = [r['tp_records'] for r in results if 'tp_records' in r and not r['tp_records'].empty]
    
    if tp_frames:
        all_tps = pd.concat(tp_frames, ignore_index=True)
        tp_export = all_tps[['DESCRIZIONE_PROGETTO']].copy()

        def clean_tp_description(value):
            text = '' if pd.isna(value) else str(value)
            text = text.replace('\r', ' ').replace('\n', ' ').replace('\t', ' ')
            text = ''.join(' ' if ord(char) > 127 else char for char in text)
            text = text.replace('"', '').replace(',', ' ').replace(';', ' ')
            return ' '.join(text.split())

        tp_export['descrizione'] = tp_export['DESCRIZIONE_PROGETTO'].apply(clean_tp_description)
        tp_export['old'] = 'ai'
        tp_export = tp_export[['descrizione', 'old']]
        tp_export = tp_export[tp_export['descrizione'] != '']
        tp_export = tp_export.drop_duplicates(subset=['descrizione'])
        
        tp_export_path = os.path.join(DATA_DIR, 'regex_tp_matches.csv')
        tp_export.to_csv(tp_export_path, index=False, quoting=3, escapechar='\\')
        print(f"[+] Salvati {len(tp_export)} True Positives (description cleaned, label=ai) in: {tp_export_path}")
    else:
        print("Nessun true positive trovato.")
else:
    print("Elaborazione non completata, risultati assenti.")

Elaborazione non completata, risultati assenti.


In [20]:
if results:
    # Aggrega tutti i dataframe dei low confidence AI trovati in ogni file
    all_low_ai = pd.concat([r['low_ai'] for r in results if 'low_ai' in r and not r['low_ai'].empty])
    
    if not all_low_ai.empty:
        # Ordina per confidenza crescente e prendi i primi 20 rimuovendo i duplicati
        lowest_ai_overall = all_low_ai.drop_duplicates(subset=['DESCRIZIONE_PROGETTO']).nsmallest(20, 'CLASSIFICAZIONE_CONFIDENZA')
        print("I 20 progetti classificati 'AI' dal modello (UNICI) con la CONFIDENZA PIÙ BASSA in assoluto:\n")
        
        for idx, row in lowest_ai_overall.iterrows():
            print(f"Confidenza: {row['CLASSIFICAZIONE_CONFIDENZA']:.4f}")
            desc = row['DESCRIZIONE_PROGETTO']
            if pd.notna(desc) and len(desc) > 500: desc = desc[:1000] + "..."
            print(f"Descrizione: {desc}")
            print("-" * 80)
    else:
        print("Nessun record AI trovato.")
else:
    print("Elaborazione non completata, risultati assenti.")

Elaborazione non completata, risultati assenti.
